# 🎵 Comparaison des 4 modèles Demucs

> **Notebook d'analyse comparative** — Séparation de sources audio avec Demucs

| Modèle | Description |
|--------|-------------|
| **htdemucs** | Hybrid Transformer Demucs (baseline) |
| **htdemucs_ft** | HTDemucs fine-tuné sur MUSDB-HQ |
| **mdx_extra** | MDX-Net (architecture convolutive) |
| **mdx_extra_q** | MDX-Net quantifié (plus léger) |

### Plan d'analyse
1. ⚙️ Configuration & imports
2. 📂 Sélection du fichier audio test
3. ⏱️ Séparation & benchmarking du temps d'inférence
4. ⏱️ Visualisation des temps
5. 🔊 Écoute des stems séparés
6. 🌊 Formes d'onde
7. 🔊 Spectrogrammes
8. 📐 Métriques spectrales avancées (RMS, centroïde, MFCC, bandes)
9. ⚡ SI-SNR
10. 📊 Métriques BSS officielles (SDR/SIR/SAR/ISR)
11. 🕸️ Vue synthèse radar
12. 🏆 Score composite & classement
13. 🎯 Verdict final + export CSV


---
## ⚙️ 1. Configuration & Imports

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import Audio, display, Markdown
import pandas as pd

# ─── Dark mode global ────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor': '#0d1117',
    'text.color': 'white', 'axes.labelcolor': 'white',
    'xtick.color': 'white', 'ytick.color': 'white',
    'axes.edgecolor': '#333333', 'grid.color': '#333333',
    'legend.facecolor': '#1a1a2e', 'legend.edgecolor': '#444444',
    'font.family': 'DejaVu Sans', 'font.size': 11,
})

# ─── Imports projet ──────────────────────────────────────────────────────────
from src.music_separation import AudioSeparator, AudioEvaluator, audio_utils, analysis_tools
from src.music_separation.config import SUPPORTED_MODELS, DEFAULT_SAMPLE_RATE
from src.music_separation.analysis_tools import (
    Timer, MODEL_COLORS, MODEL_LABELS, STEM_NAMES, STEM_COLORS,
    compute_snr, compute_si_snr, compute_rms_energy,
    compute_spectral_centroid, compute_spectral_rolloff, compute_spectral_flux,
    compute_mfcc_distance, compute_frequency_band_energy,
    plot_metrics_bar_per_stem, plot_waveforms_comparison, plot_spectrograms_grid,
    plot_frequency_bands_heatmap, plot_inference_time_comparison,
    plot_si_snr_comparison, plot_mfcc_distance_matrix,
    plot_spectral_features_comparison, plot_sdr_evolution_summary, plot_metrics_radar
)

print(f"✅ Imports OK — device : {'cuda' if __import__('torch').cuda.is_available() else 'cpu'}")
print(f"📂 Racine du projet : {PROJECT_ROOT}")

---
## 📂 2. Sélection du fichier audio & helper de sauvegarde

In [ ]:
# ── Mode de travail ───────────────────────────────────────────────────────────
USE_MUSDB = False   # ← True si tu as une vérité terrain MUSDB

# ── Option A : fichier quelconque ─────────────────────────────────────────────
AUDIO_FILE = Path(PROJECT_ROOT) / 'data' / 'input' / 'good for the ghost - Alge.mp3'

# ── Option B : morceau MUSDB ──────────────────────────────────────────────────
# MUSDB_TRACK_DIR = Path(PROJECT_ROOT) / 'dataset' / 'musdb18hq' / 'test' / 'Arise - Run Run Run'
# AUDIO_FILE = MUSDB_TRACK_DIR / 'mixture.wav'
# GT_STEMS = {
#     'vocals': MUSDB_TRACK_DIR / 'vocals.wav',
#     'drums':  MUSDB_TRACK_DIR / 'drums.wav',
#     'bass':   MUSDB_TRACK_DIR / 'bass.wav',
#     'other':  MUSDB_TRACK_DIR / 'other.wav',
# }

# ── Dossiers ──────────────────────────────────────────────────────────────────
OUTPUT_DIR  = Path(PROJECT_ROOT) / 'data' / 'output_demucs_comparison'
RESULTS_DIR = OUTPUT_DIR / 'results'
PLOTS_DIR   = RESULTS_DIR / 'plots'
for d in [OUTPUT_DIR, RESULTS_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Helper de sauvegarde ──────────────────────────────────────────────────────
def save_fig(fig: plt.Figure, name: str, dpi: int = 150) -> None:
    """
    Sauvegarde la figure dans PLOTS_DIR/<name>.png, puis l'affiche.
    Appelle toujours cette fonction AU LIEU de plt.show().
    """
    out = PLOTS_DIR / f"{name}.png"
    fig.savefig(out, dpi=dpi, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()
    print(f"  💾 Sauvegardé → {out.relative_to(PROJECT_ROOT)}")

MODELS = SUPPORTED_MODELS  # ['htdemucs', 'htdemucs_ft', 'mdx_extra', 'mdx_extra_q']

assert AUDIO_FILE.exists(), f"❌ Introuvable : {AUDIO_FILE}"
duration = audio_utils.get_duration(AUDIO_FILE)
print(f"🎵 {AUDIO_FILE.name}  |  ⏱️ {duration:.1f}s  |  Modèles : {MODELS}")
print(f"💾 Plots → {PLOTS_DIR.relative_to(PROJECT_ROOT)}")

In [ ]:
display(Markdown("### 🔈 Écoute du fichier source"))
display(Audio(str(AUDIO_FILE), autoplay=False))

---
## ⏱️ 3. Séparation avec les 4 modèles

In [ ]:
inference_times    = {}
load_times         = {}
total_times        = {}
stem_paths         = {}
stem_audio_arrays  = {}

for model_name in MODELS:
    print(f"\n{'='*60}\n  🤖 {MODEL_LABELS[model_name]}\n{'='*60}")
    model_out_dir = OUTPUT_DIR / model_name
    model_out_dir.mkdir(parents=True, exist_ok=True)

    with Timer() as t_load:
        separator = AudioSeparator(model_name=model_name)
    load_times[model_name] = t_load.elapsed
    print(f"  ⏳ Chargement : {t_load.elapsed:.2f}s")

    with Timer() as t_inf:
        saved_paths = separator.process_file(AUDIO_FILE, model_out_dir)
    inference_times[model_name] = t_inf.elapsed
    total_times[model_name]     = t_load.elapsed + t_inf.elapsed
    print(f"  ⚡ Inférence  : {t_inf.elapsed:.2f}s  |  🏁 Total : {total_times[model_name]:.2f}s")

    stem_paths[model_name] = {p.stem.split('_')[-1]: p for p in saved_paths}

print("\n✅ Séparation terminée !")

In [ ]:
print("📥 Chargement des stems en mémoire...")
for model_name in MODELS:
    stem_audio_arrays[model_name] = {}
    for stem_name, path in stem_paths[model_name].items():
        audio, _ = audio_utils.load_audio(path, sr=DEFAULT_SAMPLE_RATE, mono=False)
        stem_audio_arrays[model_name][stem_name] = audio
    print(f"  ✓ {MODEL_LABELS[model_name]} — {list(stem_audio_arrays[model_name].keys())}")

---
## ⏱️ 4. Visualisation des temps d'inférence

In [ ]:
fig = plot_inference_time_comparison(inference_times, duration)
save_fig(fig, '01_inference_times')

df_times = pd.DataFrame({
    'Modèle':         [MODEL_LABELS[m] for m in MODELS],
    'Chargement (s)': [round(load_times[m], 2) for m in MODELS],
    'Inférence (s)':  [round(inference_times[m], 2) for m in MODELS],
    'Total (s)':      [round(total_times[m], 2) for m in MODELS],
    'RTF':            [round(inference_times[m] / duration, 3) for m in MODELS],
}).set_index('Modèle')
display(Markdown("### 📋 Tableau des temps"))
display(df_times.style.background_gradient(cmap='RdYlGn_r', subset=['Inférence (s)', 'RTF']))

---
## 🔊 5. Écoute des stems séparés

In [ ]:
for stem_name in STEM_NAMES:
    display(Markdown(f"### 🎙️ **{stem_name.upper()}**"))
    for model_name in MODELS:
        path = stem_paths[model_name].get(stem_name)
        if path and path.exists():
            display(Markdown(f"**{MODEL_LABELS[model_name]}**"))
            display(Audio(str(path), autoplay=False))
    display(Markdown("---"))

---
## 🌊 6. Formes d'onde

In [ ]:
for stem_name in STEM_NAMES:
    fig = plot_waveforms_comparison(stem_audio_arrays, stem_name=stem_name,
                                    sr=DEFAULT_SAMPLE_RATE, max_seconds=15.0)
    save_fig(fig, f'02_waveform_{stem_name}')

---
## 🔊 7. Spectrogrammes comparatifs

In [ ]:
for stem_name in STEM_NAMES:
    fig = plot_spectrograms_grid(stem_audio_arrays, stem_name=stem_name, sr=DEFAULT_SAMPLE_RATE)
    save_fig(fig, f'03_spectrogram_{stem_name}')

---
## 📐 8. Métriques spectrales avancées

### 8.1 Énergie RMS

In [ ]:
rms_per_model = {
    m: {s: compute_rms_energy(a) for s, a in stem_audio_arrays[m].items()}
    for m in MODELS
}

df_rms = pd.DataFrame(rms_per_model).T.round(5)
df_rms.index = [MODEL_LABELS[m] for m in df_rms.index]
display(Markdown("### 📋 Énergie RMS par stem"))
display(df_rms.style.background_gradient(cmap='Blues', axis=None))

# ── Pie charts ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(MODELS), figsize=(20, 5))
fig.patch.set_facecolor('#0d1117')
for ax, model_name in zip(axes, MODELS):
    rms = rms_per_model[model_name]
    stems_present = [s for s in STEM_NAMES if s in rms]
    vals   = [rms[s] for s in stems_present]
    colors = [STEM_COLORS.get(s, 'gray') for s in stems_present]
    wedges, texts, autotexts = ax.pie(
        vals, labels=[s.capitalize() for s in stems_present],
        colors=colors, autopct='%1.1f%%', startangle=90,
        wedgeprops=dict(edgecolor='white', linewidth=1.2)
    )
    for t in texts:      t.set_color('white');  t.set_fontsize(10)
    for at in autotexts: at.set_color('black'); at.set_fontsize(9)
    ax.set_facecolor('#0d1117')
    ax.set_title(MODEL_LABELS[model_name], color='white', fontsize=11)
fig.suptitle("Distribution d'énergie RMS", color='white', fontsize=14)
plt.tight_layout()
save_fig(fig, '04_rms_distribution')

### 8.2 Caractéristiques spectrales (Centroïde, Roll-off, Flux)

In [ ]:
print("📐 Calcul des features spectrales...")
spectral_features = {}
for model_name in MODELS:
    spectral_features[model_name] = {}
    for stem_name, audio in stem_audio_arrays[model_name].items():
        spectral_features[model_name][stem_name] = {
            'centroid': compute_spectral_centroid(audio, sr=DEFAULT_SAMPLE_RATE),
            'rolloff':  compute_spectral_rolloff(audio,  sr=DEFAULT_SAMPLE_RATE),
            'flux':     compute_spectral_flux(audio,     sr=DEFAULT_SAMPLE_RATE),
        }
    print(f"  ✓ {MODEL_LABELS[model_name]}")

fig = plot_spectral_features_comparison(spectral_features, STEM_NAMES)
save_fig(fig, '05_spectral_features')

df_centroid = pd.DataFrame({
    m: {s: round(spectral_features[m][s]['centroid'], 1)
        for s in STEM_NAMES if s in spectral_features[m]}
    for m in MODELS
}).T.rename(index=MODEL_LABELS)
display(Markdown("### 📋 Centroïde spectral (Hz)"))
display(df_centroid.style.background_gradient(cmap='cool', axis=None))

### 8.3 Énergie par bandes de fréquences

In [ ]:
print("📊 Calcul de l'énergie par bandes...")
band_energies = {}
for model_name in MODELS:
    band_energies[model_name] = {}
    for stem_name, audio in stem_audio_arrays[model_name].items():
        band_energies[model_name][stem_name] = compute_frequency_band_energy(audio, sr=DEFAULT_SAMPLE_RATE)
    print(f"  ✓ {MODEL_LABELS[model_name]}")

for stem_name in STEM_NAMES:
    if all(stem_name in band_energies[m] for m in MODELS):
        fig = plot_frequency_bands_heatmap(band_energies, stem_name=stem_name)
        save_fig(fig, f'06_freq_bands_{stem_name}')

### 8.4 Distance MFCC

In [ ]:
print("🎼 Calcul des distances MFCC (ref = htdemucs)...")
ref_model = 'htdemucs'
mfcc_distances = {}
for model_name in MODELS:
    mfcc_distances[model_name] = {}
    for stem_name in STEM_NAMES:
        ref = stem_audio_arrays.get(ref_model, {}).get(stem_name)
        est = stem_audio_arrays.get(model_name, {}).get(stem_name)
        if ref is not None and est is not None:
            mfcc_distances[model_name][stem_name] = compute_mfcc_distance(ref, est, sr=DEFAULT_SAMPLE_RATE)
    print(f"  ✓ {MODEL_LABELS[model_name]}")

fig = plot_mfcc_distance_matrix(mfcc_distances, stem_names=STEM_NAMES)
save_fig(fig, '07_mfcc_distance')

df_mfcc = pd.DataFrame(mfcc_distances).T.round(2).rename(index=MODEL_LABELS)
display(Markdown(f"### 📋 Distance MFCC vs {MODEL_LABELS[ref_model]} (↓ meilleur)"))
display(df_mfcc.style.background_gradient(cmap='RdYlGn_r', axis=None))

---
## ⚡ 9. SI-SNR (Scale-Invariant SNR)

In [ ]:
print("⚡ Calcul des SI-SNR (ref = htdemucs)...")
ref_model = 'htdemucs'
si_snr_per_model = {}
snr_per_model    = {}
for model_name in MODELS:
    si_snr_per_model[model_name] = {}
    snr_per_model[model_name]    = {}
    for stem_name in STEM_NAMES:
        ref = stem_audio_arrays.get(ref_model, {}).get(stem_name)
        est = stem_audio_arrays.get(model_name, {}).get(stem_name)
        if ref is not None and est is not None:
            si_snr_per_model[model_name][stem_name] = compute_si_snr(ref, est)
            snr_per_model[model_name][stem_name]    = compute_snr(ref, est)
    print(f"  ✓ {MODEL_LABELS[model_name]}")

fig = plot_si_snr_comparison(si_snr_per_model, stem_names=STEM_NAMES)
save_fig(fig, '08_si_snr')

df_sisnr = pd.DataFrame(si_snr_per_model).T.round(2).rename(index=MODEL_LABELS)
display(Markdown("### 📋 SI-SNR par stem (dB) — ↑ meilleur"))
display(df_sisnr.style
        .background_gradient(cmap='RdYlGn', axis=None)
        .highlight_max(color='#1a472a', axis=0)
        .highlight_min(color='#7d1111', axis=0))

---
## 📊 10. Métriques BSS (SDR / SIR / SAR / ISR)
> Activé uniquement si `USE_MUSDB = True`.

In [ ]:
bss_results = {}
HAS_BSS = False

if not USE_MUSDB:
    display(Markdown("> ℹ️ **Mode sans vérité terrain** — BSS désactivées. Passez `USE_MUSDB=True`."))
else:
    from src.music_separation import AudioEvaluator
    evaluator = AudioEvaluator(sample_rate=DEFAULT_SAMPLE_RATE)
    gt_paths  = [GT_STEMS[s] for s in STEM_NAMES]
    for model_name in MODELS:
        print(f"\n📊 {MODEL_LABELS[model_name]}...")
        pred_paths = [p for p in [stem_paths[model_name].get(s) for s in STEM_NAMES] if p]
        try:
            metrics = evaluator.compute_bss_metrics(gt_paths, pred_paths)
            bss_results[model_name] = metrics
            print({k: np.round(v,2) for k,v in metrics.items()})
        except Exception as e:
            print(f"  ❌ {e}")
    HAS_BSS = bool(bss_results)
    print("\n✅ BSS terminées !")

In [ ]:
if HAS_BSS:
    for metric_name in ['SDR', 'SIR', 'SAR', 'ISR']:
        if any(metric_name in bss_results[m] for m in bss_results):
            fig = plot_metrics_bar_per_stem(bss_results, metric_name=metric_name, stem_names=STEM_NAMES)
            save_fig(fig, f'09_bss_{metric_name.lower()}')

    fig = plot_sdr_evolution_summary(bss_results, stem_names=STEM_NAMES)
    save_fig(fig, '09_sdr_summary')

    df_sdr = pd.DataFrame({
        MODEL_LABELS[m]: {f"{s.capitalize()} SDR": round(float(bss_results[m]['SDR'][i]), 2)
                          for i, s in enumerate(STEM_NAMES) if i < len(bss_results[m]['SDR'])}
        for m in bss_results
    })
    display(Markdown("### 📋 SDR par stem"))
    display(df_sdr.style
            .background_gradient(cmap='RdYlGn', axis=1)
            .highlight_max(color='#1a472a', axis=1)
            .highlight_min(color='#7d1111', axis=1))

---
## 🕸️ 11. Vue Synthèse — Radar

In [ ]:
def norm(values: dict, higher_is_better: bool = True) -> dict:
    vals = np.array(list(values.values()), dtype=float)
    vmin, vmax = np.nanmin(vals), np.nanmax(vals)
    if vmax == vmin:
        return {m: 0.5 for m in values}
    normed = (vals - vmin) / (vmax - vmin)
    if not higher_is_better:
        normed = 1 - normed
    return {m: float(normed[i]) for i, m in enumerate(values)}

sisnr_global = {m: np.nanmean(list(si_snr_per_model[m].values())) for m in MODELS}
flux_global  = {m: np.nanmean([spectral_features[m][s]['flux'] for s in STEM_NAMES if s in spectral_features[m]]) for m in MODELS}
bass_energy  = {m: band_energies[m].get('bass', {}).get('bass (80-250 Hz)', 0.0) for m in MODELS}
vocals_mid   = {m: band_energies[m].get('vocals', {}).get('midrange (250-2kHz)', 0.0) for m in MODELS}
mfcc_global  = {m: np.nanmean(list(mfcc_distances[m].values())) if mfcc_distances[m] else 0.0 for m in MODELS}

radar_metrics = {}
for m in MODELS:
    radar_metrics[m] = {
        'SI-SNR':                norm(sisnr_global, True)[m],
        'Vitesse':               norm(inference_times, False)[m],
        'Propreté\nspectrale':   norm(flux_global, False)[m],
        'Clarté\nbasses':        norm(bass_energy, True)[m],
        'Clarté\nvoix':          norm(vocals_mid, True)[m],
        'Similarité\ntimbrale':  norm(mfcc_global, False)[m],
    }

fig = plot_metrics_radar(radar_metrics, title="Comparaison radar (normalisée 0–1)")
save_fig(fig, '10_radar')

df_radar = pd.DataFrame(radar_metrics).T.round(3).rename(index=MODEL_LABELS)
display(Markdown("### 📋 Scores normalisés (0 = pire, 1 = meilleur)"))
display(df_radar.style
        .background_gradient(cmap='RdYlGn', axis=None)
        .highlight_max(color='#1a472a', axis=0)
        .highlight_min(color='#7d1111', axis=0))

---
## 🏆 12. Score composite & classement

In [ ]:
weights = {
    'SI-SNR':               0.25,
    'Vitesse':              0.20,
    'Propreté\nspectrale':  0.15,
    'Clarté\nbasses':       0.15,
    'Clarté\nvoix':         0.10,
    'Similarité\ntimbrale': 0.15,
}

final_scores = {m: round(sum(radar_metrics[m][k] * w for k, w in weights.items()), 4) for m in MODELS}
ranked = sorted(final_scores.items(), key=lambda x: x[1], reverse=True)

medals = ['🥇', '🥈', '🥉', '4️⃣']
print("\n" + "="*55 + "\n  🏆 CLASSEMENT FINAL\n" + "="*55)
for i, (model, score) in enumerate(ranked):
    print(f"  {medals[i]} {MODEL_LABELS[model]:<20} → {score:.4f}")
print("="*55)

# ── Graphique barres finales ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#0d1117')
sorted_models = [m for m, _ in ranked]
scores  = [final_scores[m] for m in sorted_models]
colors  = [MODEL_COLORS[m]  for m in sorted_models]
labels  = [MODEL_LABELS[m]  for m in sorted_models]
bars = ax.barh(labels[::-1], scores[::-1], color=colors[::-1],
               edgecolor='white', linewidth=0.5, alpha=0.88)
for bar, s in zip(bars, scores[::-1]):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f"{s:.3f}", va='center', ha='left', color='white', fontsize=11, fontweight='bold')
ax.set_xlabel('Score composite pondéré', color='white', fontsize=12)
ax.set_title('🏆 Classement Final', color='white', fontsize=14)
ax.tick_params(colors='white')
ax.spines[:].set_color('#333')
ax.xaxis.grid(True, color='#333', linestyle='--', linewidth=0.8)
ax.set_xlim(0, max(scores) * 1.15)
plt.tight_layout()
save_fig(fig, '11_final_ranking')

---
## 🎯 13. Verdict final & export

In [ ]:
best_model    = ranked[0][0]
fastest_model = min(inference_times, key=inference_times.get)
most_similar  = min(
    [m for m in MODELS if m != 'htdemucs'],
    key=lambda m: np.nanmean(list(mfcc_distances[m].values())) if mfcc_distances[m] else float('inf'),
    default=ranked[1][0]
)

verdict = f"""
## 🎯 Verdict Final

### 🏆 Meilleur modèle global
**{MODEL_LABELS[best_model]}** — score composite **{final_scores[best_model]:.3f}**

### ⚡ Modèle le plus rapide
**{MODEL_LABELS[fastest_model]}** — {inference_times[fastest_model]:.1f}s (RTF = {inference_times[fastest_model]/duration:.2f}×)

### 🎼 Plus proche de HTDemucs (MFCC)
**{MODEL_LABELS[most_similar]}**

### 📊 Classement
| Rang | Modèle | Score |
|------|--------|-------|
""" + "\n".join(f"| {i+1} | {MODEL_LABELS[m]} | {s:.3f} |" for i,(m,s) in enumerate(ranked))

verdict += f"""

### 💡 Recommandations
| Contexte | Modèle | Raison |
|----------|--------|--------|
| Production (qualité max) | {MODEL_LABELS[best_model]} | Meilleur score global |
| Temps réel / edge | {MODEL_LABELS[fastest_model]} | Le plus rapide |
| Équilibre qualité/vitesse | HTDemucs | Robuste par défaut |
| GPU limité | MDX Extra Q | Modèle quantifié |
"""
display(Markdown(verdict))

In [ ]:
# ── Export CSV ────────────────────────────────────────────────────────────────
df_times.to_csv(RESULTS_DIR / 'inference_times.csv')
df_radar.to_csv(RESULTS_DIR / 'normalized_scores.csv')
df_rms.to_csv(RESULTS_DIR / 'rms_energy.csv')
df_mfcc.to_csv(RESULTS_DIR / 'mfcc_distances.csv')
df_sisnr.to_csv(RESULTS_DIR / 'si_snr.csv')
pd.DataFrame(list(final_scores.items()), columns=['model', 'score']) \
  .assign(model=lambda df: df['model'].map(MODEL_LABELS)) \
  .sort_values('score', ascending=False).set_index('model') \
  .to_csv(RESULTS_DIR / 'final_scores.csv')

print(f"✅ CSV  → {RESULTS_DIR.relative_to(PROJECT_ROOT)}/")
print(f"✅ PNG  → {PLOTS_DIR.relative_to(PROJECT_ROOT)}/")
print(f"   {len(list(PLOTS_DIR.glob('*.png')))} figures sauvegardées")

---
> **Projet IA — ENSTA Paris 2025/2026** | [Demucs](https://github.com/facebookresearch/demucs) (Meta AI Research)